In [ ]:
import scanpy as sc
import anndata as ad
import scipy

from rpy2.robjects import pandas2ri


import anndata2ri

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
adata_adt = sc.AnnData(X = adata.obsm['protein_counts'])
# del adata.obsm
# del adata.obsp

In [ ]:
%%R
suppressPackageStartupMessages({
    library(SingleCellExperiment)
    library(Seurat)
    library(Signac)
})

In [5]:
adata_ = ad.AnnData(adata_gex.X)
adata_.obs_names = adata_gex.obs_names
adata_.var_names = adata_gex.var_names
adata_.obs['cell_type'] = adata_gex.obs['cell_type']
adata_.obs['batch'] = adata_gex.obs['batch']

In [ ]:
%%R -i adata_
rna = as.Seurat(adata_, counts='X', data=NULL)
rna <- RenameAssays(rna, originalexp="RNA")
rna.list <- SplitObject(rna, split.by = "batch")
rna.list <- lapply(X = rna.list, FUN = SCTransform, variable.features.n = 2000)
features <- SelectIntegrationFeatures(object.list = rna.list, nfeatures = 2000)
rna.list <- PrepSCTIntegration(object.list = rna.list, anchor.features = features)
anchors <- FindIntegrationAnchors(object.list = rna.list, normalization.method = "SCT", anchor.features = features)
integrated <- IntegrateData(anchorset = anchors, normalization.method = "SCT")
integrated <- RunPCA(integrated)


In [7]:
adata_ = ad.AnnData(adata_atac.X)
adata_.obs_names = adata_atac.obs_names
adata_.var_names = adata_atac.var_names
adata_.obs['cell_type'] = adata_atac.obs['cell_type']
adata_.obs['batch'] = adata_atac.obs['batch']

In [ ]:
%%R -i adata_
atac = as.Seurat(adata_, counts='X', data=NULL)
atac <- RenameAssays(atac, originalexp ='ATAC')
atac.list <- SplitObject(atac, split.by = "batch")

atac.list <- lapply(X = atac.list, FUN = function(X) {
  DefaultAssay(X) <- "ATAC"
  X <- FindTopFeatures(X, min.cutoff = 10)
  X <- RunTFIDF(X)
  X <- RunSVD(X)
})
combined <- merge(x = atac.list[[1]], y= atac.list[2:length(atac.list)] )

combined <- FindTopFeatures(combined, min.cutoff = 10)
combined <- RunTFIDF(combined)
combined <- RunSVD(combined)

integration.anchors <- FindIntegrationAnchors(
  object.list = atac.list,
  anchor.features = rownames(combined),
  reduction = "rlsi",
  dims = 2:30
)

# integrate LSI embeddings
integrated_atac <- IntegrateEmbeddings(
  anchorset = integration.anchors,
  reductions = combined[["lsi"]],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = 1:30
)

In [9]:
adata_ = ad.AnnData(adata_adt.X)
adata_.obs_names = adata_adt.obs_names
adata_.var_names = adata_adt.var_names
adata_.obs['cell_type'] = adata_adt.obs['cell_type']
adata_.obs['batch'] = adata_adt.obs['batch']

In [ ]:
%%R -i adata_
cite = as.Seurat(adata_, counts='X', data=NULL)
cite

cite <- RenameAssays(cite, originalexp='ADT')

cite.list <- SplitObject(cite, split.by = "batch")

cite.list <- lapply(X = cite.list, FUN = function(x) {
    VariableFeatures(x) <- rownames(x[["ADT"]])
    x <- NormalizeData(x, normalization.method = 'CLR', margin = 2, verbose=FALSE)
})

features <- SelectIntegrationFeatures(object.list = cite.list)

cite.list <- lapply(X = cite.list, FUN = function(x) {
    x <- ScaleData(x, features = features, verbose=FALSE)
    x <- RunPCA(x, features = features, reduction.name = "pca", verbose=FALSE)
})

anchors <- FindIntegrationAnchors(object.list = cite.list, reduction = "rpca", 
    dims = 1:30, verbose=FALSE)
integrated_adt <- IntegrateData(anchorset = anchors, dims = 1:30)

integrated_adt <- ScaleData(integrated_adt, verbose=FALSE)
integrated_adt <- RunPCA(integrated_adt, reduction.name = "apca", verbose=FALSE)

In [11]:
%%R
integrated[['ATAC']] = integrated_atac[['ATAC']]
integrated[["Iatac"]] <- integrated_atac[["integrated_lsi"]]
integrated[["IADT"]] <- integrated_adt[["integrated"]]
integrated[["apca"]] <- integrated_adt[["apca"]]

integrated <- FindMultiModalNeighbors(integrated, reduction.list = list("pca", "Iatac", 'apca'), 
                                      dims.list = list(1:50, 1:30, 1:30))
# 
integrated <- RunSPCA(integrated, assay = 'integrated', graph = 'wsnn')


  |                                                  | 0 % ~calculating   |+++++++++++++++++                                 | 33% ~20s           |++++++++++++++++++++++++++++++++++                | 67% ~10s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=29s  
  |                                                  | 0 % ~calculating   |+++++++++++++++++                                 | 33% ~04s           |++++++++++++++++++++++++++++++++++                | 67% ~01s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=04s  
  |                                                  | 0 % ~calculating   |+++++++++++++++++                                 | 33% ~01m 05s       |++++++++++++++++++++++++++++++++++                | 67% ~34s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=01m 44s
  |                                                  | 0 % ~calculating   |+++++++++++++++++                                 | 3

Calculating cell-specific modality weights
Finding 20 nearest neighbors for each modality.
Calculating kernel bandwidths
Finding multimodal neighbors
Constructing multimodal KNN graph
Constructing multimodal SNN graph
Computing sPCA transformation
In addition: Warning messages:
1: Key ‘integrated_’ taken, using ‘iadt_’ instead 
2: Key ‘PC_’ taken, using ‘apca_’ instead 


In [12]:
%%R -o spca
spca = Embeddings(object = integrated[["spca"]])

In [13]:
adata = sc.AnnData(spca)
adata.obs = adata_.obs
adata

AnnData object with n_obs × n_vars = 25517 × 50
    obs: 'cell_type', 'batch'

In [14]:
%%R -o wnn
wnn <- as.data.frame(summary(integrated@graphs$wknn))

In [ ]:
wnn['i'] = wnn['i'] - 1
wnn['j'] = wnn['j'] - 1
adata.obsp['wnn_connectivities'] = scipy.sparse.coo_matrix((wnn['x'], (wnn['i'], wnn['j'])))
adata.obsp['wnn_connectivities'] = scipy.sparse.csr_matrix(adata.obsp['wnn_connectivities'])
adata.write('Seurat_tea_3.h5ad')